In [1]:
from pathlib import Path
import os
import re

### Preparation

The Boltz calculation requires two parts, a **multi-sequence alignment (MSA)** step and an **inference step**.  

First we need to define where Boltz-2 finds our input data and where the output files are written to. You can see these files in the file browser on the left. If you change these names, remember to change them in the analysis notebook as well.

In [2]:
BOLTZ_WORKING_DIR = Path.home() / "boltz_test"

### MSA Calculation

Now we can start with the MSA. 
For the MSA calculation we need the sequence in `fasta` format. Feel free to look at our example to adapt your input. For each protein entity in the Boltz simulation the MSA needs to be computed individually beforehand!

Here we take human insulin from UniProt:
[https://www.uniprot.org/uniprotkb/P01308/entry](https://www.uniprot.org/uniprotkb/P01308/entry)

In [3]:
INPUT_FASTA = BOLTZ_WORKING_DIR / "insulin.fasta"

Next we can prepare the input file for the colabfold run that calls mmseqs2.

In [4]:
MSA_FILE = BOLTZ_WORKING_DIR / "run_msa.sh"
db_dir = "/mnt/sds-hd/sd25g005/boltz/localcolabfold"

test_file = f'''
#!/bin/bash

module load devel/miniforge/24.9.2
module load devel/cuda/12.8
conda activate /mnt/sds-hd/sd25g005/boltz

export PATH="/mnt/sds-hd/sd25g005/boltz/localcolabfold/colabfold-conda/bin:$PATH"

colabfold_search  \\
    --db-load-mode 2 \\
    --threads 96 \\
    --use-env 0 \\
    --gpu 1 \\
    "{INPUT_FASTA}" \\
    "{db_dir}" \\
    "{BOLTZ_WORKING_DIR}" \\
    
'''

with open(MSA_FILE, "w") as text_file:
    text_file.write(test_file)

Execute the cell below to start the MSA prediction. This takes about 10 minutes for insulin.

In [5]:
os.system(f'echo "Running file {MSA_FILE}"')
os.system(f"bash {MSA_FILE} > colabfold_search.log 2>&1")
os.system(f'echo "Done with {MSA_FILE}"')

Running file /home/hd/hd_hd/hd_aq354/boltz_test/run_msa.sh
Done with /home/hd/hd_hd/hd_aq354/boltz_test/run_msa.sh


0

### Inference run
Now we need to prepare the `.yaml` input file that will be tell Boltz what to predict.

More information and examples on how these files are structured can be found in the [Boltz github](https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md#input-format).

Important parameters in the input file are the `name`, `sequence` and `id`. We also link the MSA that we just calculated. Upon executing this cell, the input file will be written to your working directory.

Remember the name of your input file as it is needed for the analysis later! 

In [6]:
INPUT_FILE = BOLTZ_WORKING_DIR / "insulin.yaml"

A3M_FILE = (
    re.sub(r"[ |> =]", "_", open(INPUT_FASTA).readline().lstrip(">").strip()) + ".a3m"
)
MSA_PATH = BOLTZ_WORKING_DIR / A3M_FILE

test_file = f"""
version: 1
sequences:
  - protein:
      id: [A] 
      sequence: MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN
      msa: {MSA_PATH}
"""

with open(INPUT_FILE, "w") as text_file:
    text_file.write(test_file)

Next we need to write the run file, which loads all relevant modules and handles the Boltz .yaml file in a program call. You do not need to change these parameters. A full list is available here. Only change the parameters if you know what you are doing.


In [7]:
BOLTZ_RUN_FILE = BOLTZ_WORKING_DIR / "run.sh"

run_file = f"""
#!/bin/bash

module load devel/miniforge/24.9.2
module load devel/cuda/12.8
conda activate /mnt/sds-hd/sd25g005/boltz

boltz predict  {str(INPUT_FILE)}  \\
    --write_full_pae \\
    --out_dir {BOLTZ_WORKING_DIR}
"""

with open(BOLTZ_RUN_FILE, "w") as file:
    file.write(run_file)

### Run the Boltz prediction

Execute the next cell to star the run! Good luck!

In [8]:
os.system(f'echo "Running file {BOLTZ_RUN_FILE}"')
os.system(f"bash {BOLTZ_RUN_FILE}")

Running file /home/hd/hd_hd/hd_aq354/boltz_test/run.sh
Checking input data.
Processing 1 inputs with 1 threads.


100%|██████████| 1/1 [00:00<00:00,  5.88it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Running structure prediction for 1 input.


/home/hd/hd_hd/hd_aq354/.local/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
You are using a CUDA device ('NVIDIA A40') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.
/home/hd/hd_hd/hd_aq354/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 2 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessiv

Predicting DataLoader 0: 100%|██████████| 1/1 [00:14<00:00,  0.07it/s]


0

Done! 

Number of failed examples should be 0. Also check if there is a warning "MSA does not match input sequence" that indicates that something went wrong with the precomputed msa.